# 03 — Train Neural Collaborative Filtering (NCF) on Colab GPU

This notebook trains the NeuMF model (from `src/models/ncf.py`) on the full MovieLens 25M dataset using Colab's free T4 GPU. Training takes ~15–20 minutes.

**Architecture:** NeuMF = GMF branch (linear) + MLP branch (nonlinear) fused via sigmoid. See `src/models/ncf.py` and the paper: https://arxiv.org/abs/1708.05031

**Output:** `ncf.pth` checkpoint, downloaded locally and (optionally) uploaded to a GitHub Release for use by `make evaluate`.

## 1. Setup

In [ ]:
import os, sys, time, urllib.request, zipfile
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__}  device={device}')
gpu_count = torch.cuda.device_count()
print(f'GPUs available: {gpu_count}')
for g in range(gpu_count):
    print(f'  GPU {g}: {torch.cuda.get_device_name(g)}')

## 2. Clone the repo

Clones the project so we import the SAME model code that `make evaluate` uses locally. No copy-paste drift.

In [ ]:
if not os.path.isdir('recsys-engine'):
    !git clone https://github.com/its-Sohan/recsys-engine.git
sys.path.insert(0, 'recsys-engine')

from src.models.ncf import NeuMF
print('NeuMF imported from local repo')

## 3. Download MovieLens 25M

Direct download in Colab (~250MB, ~1 min).

In [ ]:
DATA_DIR = 'ml-25m'
if not os.path.isdir(DATA_DIR):
    url = 'https://files.grouplens.org/datasets/movielens/ml-25m.zip'
    print('Downloading...')
    urllib.request.urlretrieve(url, 'ml-25m.zip')
    with zipfile.ZipFile('ml-25m.zip') as z:
        z.extractall('.')
    os.remove('ml-25m.zip')
print('Dataset ready:', os.listdir(DATA_DIR))

## 4. Load + time-based split

Same split logic as `src/data/loader.py` — test = latest 20% by timestamp. We only need train for NCF; eval happens locally.

In [ ]:
ratings = pd.read_csv(f'{DATA_DIR}/ratings.csv',
    dtype={'userId': np.int32, 'movieId': np.int32, 'rating': np.float32, 'timestamp': np.int64})
ratings = ratings.sort_values('timestamp').reset_index(drop=True)
n_test = int(len(ratings) * 0.2)
train = ratings.iloc[:-n_test].copy()
test  = ratings.iloc[-n_test:].copy()
print(f'train: {len(train):,}  test: {len(test):,}')

## 5. Streaming negative-sampling dataset (low RAM)

Instead of precomputing ALL negatives in memory (~125M tuples = 3GB), this generates negatives on-the-fly per `__getitem__` call. Stores only positives (~25M, ~400MB). Standard approach for large-scale NCF training.

In [ ]:
# Build id maps
users = np.sort(train.userId.unique())
items = np.sort(train.movieId.unique())
user2idx = {u: i for i, u in enumerate(users)}
item2idx = {it: i for i, it in enumerate(items)}
idx2item = {i: it for it, i in item2idx.items()}
print(f'{len(user2idx):,} users, {len(item2idx):,} items')


class StreamingNCFDataset(Dataset):
    def __init__(self, positives, user2idx, item2idx, n_items, neg_ratio=4, seed=42):
        self.pos_users = positives['userId'].map(user2idx).to_numpy(dtype=np.int64)
        self.pos_items = positives['movieId'].map(item2idx).to_numpy(dtype=np.int64)
        self.n_pos = len(self.pos_users)
        self.neg_ratio = neg_ratio
        self.n_items = n_items
        self.rng = np.random.default_rng(seed)
        seen = positives.groupby('userId')['movieId'].apply(set).to_dict()
        self.user_seen_idx = {
            user2idx[u]: {item2idx[i] for i in seen.get(u, set()) if i in item2idx}
            for u in user2idx
        }

    def __len__(self):
        return self.n_pos * (1 + self.neg_ratio)

    def __getitem__(self, idx):
        if idx < self.n_pos:
            u = int(self.pos_users[idx])
            i = int(self.pos_items[idx])
            label = 1
        else:
            pos_idx = idx // (1 + self.neg_ratio)
            u = int(self.pos_users[pos_idx])
            seen_set = self.user_seen_idx.get(u, set())
            i = int(self.rng.integers(0, self.n_items))
            for _ in range(5):
                if i not in seen_set:
                    break
                i = int(self.rng.integers(0, self.n_items))
            label = 0
        return np.int64(u), np.int64(i), np.float32(label)


positives = train[train['rating'] >= 4.0]
dataset = StreamingNCFDataset(positives, user2idx, item2idx, n_items=len(item2idx))
loader = DataLoader(dataset, batch_size=1024 * max(1, gpu_count),
                    shuffle=True, num_workers=2)
print(f'training tuples: {len(dataset):,}  (streamed, not precomputed)')

## 5.5 — Multi-GPU DataParallel setup

If 2+ GPUs are available, wraps the model in `DataParallel` so each forward/backward batch is split across all GPUs automatically. Batch size is scaled linearly per GPU.

In [ ]:
model = NeuMF(len(user2idx), len(item2idx))

if gpu_count > 1:
    model = torch.nn.DataParallel(model, device_ids=list(range(gpu_count)))
    print(f'Wrapped model for DataParallel on {gpu_count} GPUs')

model = model.to(device)

## 6. Train NeuMF on GPU(s)

~15 min for 15 epochs on 2× T4. Loss should drop from 0.6931 (= ln2, random) to ~0.30–0.40.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCELoss()

EPOCHS = 15
model.train()
for epoch in range(EPOCHS):
    t0 = time.time()
    total, n = 0.0, 0
    for bu, bi, bl in loader:
        bu, bi, bl = bu.to(device), bi.to(device), bl.to(device)
        optimizer.zero_grad()
        preds = model(bu, bi)
        loss = criterion(preds, bl)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(bu)
        n += len(bu)
    print(f'epoch {epoch+1:2d}/{EPOCHS}  loss={total/n:.4f}  ({time.time()-t0:.0f}s)')

## 7. Save the `.pth` checkpoint

DataParallel wraps the model. We unwrap via `.module` to save only the real NeuMF state_dict — `NCFRecommender.load()` doesn't use DataParallel.

In [ ]:
if hasattr(model, 'module'):
    state_dict = model.module.state_dict()
else:
    state_dict = model.state_dict()

seen = train.groupby('userId')['movieId'].apply(set).to_dict()
payload = {
    'state_dict': state_dict,
    'config': {'gmf_dim': 64, 'mlp_embed_dim': 32, 'mlp_layers': [64, 32, 16, 8]},
    'user2idx': user2idx,
    'idx2item': {int(k): int(v) for k, v in idx2item.items()},
    'seen': {int(k): list(v) for k, v in seen.items()},
}
torch.save(payload, 'ncf.pth')
print(f'Saved ncf.pth ({os.path.getsize("ncf.pth")/1e6:.1f} MB)')
print('\nDownloading to your machine...')
from google.colab import files
files.download('ncf.pth')

## 8. (Optional) Upload to a GitHub Release

If you'd rather not manage the file locally, upload it as a GitHub Release asset so the Dockerfile can fetch it at build time. Create a release on GitHub (tag: `ncf-v1`), then:

```
# Upload via GitHub web UI: Releases -> ncf-v1 -> Upload asset.
```

After downloading, place the file at `artifacts/ncf.pth` locally and run:
```
python -m src.models.train --load-ncf artifacts/ncf.pth
make evaluate
```

## What just happened

1. **GMF branch** learned linear user-item interactions (generalized dot product).
2. **MLP branch** learned nonlinear interactions via stacked dense layers.
3. **NeuMF fusion** learned how to weight both branches via a final sigmoid.
4. **Streaming negative sampling** avoided storing 125M tuples in RAM.
5. **DataParallel** split the workload across GPUs.

Loss should have dropped from 0.693 (ln 2 = random guessing) to ~0.30–0.40.